# Kernel

In [ ]:
import re
import copy
import requests

from llm_sandbox_ui.app_config import KERNEL_URL


In [ ]:

def strip_ansi(text):
    """Remove ANSI escape codes from text.
    
    Args:
        text: String potentially containing ANSI color/formatting codes.
        
    Returns:
        String with all ANSI escape sequences removed.
    """
    return re.sub(r'\x1b\[[0-9;]*m', '', text)
    

def format_kernel_stream(json_response, clean_ansi=True ):
    """Format raw kernel response into clean, consolidated messages.
    
    Merges consecutive stream messages of the same type and optionally
    strips ANSI codes from error tracebacks.
    
    Args:
        json_response: Dict with 'messages' key containing kernel output.
        clean_ansi: If True, remove ANSI codes from tracebacks.
        
    Returns:
        List of formatted message dicts with consolidated streams.
    """    
    formatted_response = []
    prev_name = None
    prev_type = None
    
    for msg in json_response.get('messages', []):
        msg_copy = copy.deepcopy(msg)
        
        if msg['msg_type'] == 'error':
            if clean_ansi:
                tb = msg_copy['content']['traceback']
                msg_copy['content']['traceback'] = ['\n'.join([strip_ansi(line) for line in tb])]

        appended = False
        if msg['msg_type'] == 'stream':
            if prev_type == msg['msg_type'] and prev_name == msg['content']['name']:
                formatted_response[-1]['content']['text'] += msg['content']['text']
                appended = True

        if not appended:
            formatted_response.append(msg_copy)

        prev_type = msg['msg_type']
        if msg['msg_type'] == 'stream':
            prev_name = msg['content']['name']

    return formatted_response


def execute_unreal_code(code):
    """ 
    Execute Python code in the Unreal Engine kernel via ngrok.
    
    Args:
        code (str): Python code to execute in Unreal
        
    Returns:
        list: Formatted kernel messages containing:
            - 'stream' messages with stdout/stderr output
            - 'error' messages with exception name, value, and cleaned traceback
    """
    
    try:
        response = requests.post(
            f'{KERNEL_URL}/execute_sync',
            json={'code': code},
            timeout=30
        )
        json_response = response.json()

    except (requests.RequestException, requests.Timeout) as e:
        # Return error in same format as kernel errors
        error_msg = [{"msg_type": "error", "content": {"ename": "KernelError", "evalue": str(e), "traceback": []}}]
        return error_msg, error_msg

    print (json_response)
    return format_kernel_stream(json_response), format_kernel_stream(json_response,clean_ansi=False)

def convert_to_accumulated(json_response):
    """
    Convert Python kernel response messages to JavaScript accumulated output format.
    
    Transforms a list of Jupyter kernel messages into the format expected by the
    frontend's output rendering system. Merges consecutive stream outputs of the
    same type (stdout/stderr).
    
    Args:
        json_response: List of dicts with 'msg_type' and 'content' keys from unreal_llm_sandbox.kernel.
    
    Returns:
        List of accumulated output dicts with 'output_type' and type-specific fields.
    """
    accumulated = []
    for msg in json_response:
        t = msg['msg_type']
        c = msg['content']
        if t == 'stream':
            # Check if we can merge with previous
            if accumulated and accumulated[-1].get('output_type') == 'stream' and accumulated[-1].get('name') == c['name']:
                accumulated[-1]['text'].append(c['text'])
            else:
                accumulated.append({'output_type': 'stream',
                                    'name': c['name'],
                                    'text': [c['text']]})
        elif t == 'execute_result':
            accumulated.append({'output_type': 'execute_result',
                                 'data': c['data'],
                                 'execution_count': c.get('execution_count')})
        elif t == 'error':
            accumulated.append({'output_type': 'error', 
                                'ename': c['ename'],
                                'evalue': c['evalue'],
                                'traceback': c['traceback']})
    return accumulated
    